# Exploratory Data Analysis — Home Credit Default Risk

Initial exploration of the `application_train.csv` dataset from the
[Home Credit Default Risk](https://www.kaggle.com/competitions/home-credit-default-risk)
Kaggle competition.

**Goal:** understand the data well enough to train a baseline model and
design the input schema of the prediction API. The focus of this project
is deployment, not model performance, so this analysis is intentionally brief.

**Dataset:** 307,511 loan applications, 122 columns. The target (`TARGET`)
is 1 if the client had payment difficulties and 0 otherwise.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/application_train.csv")
df.shape

(307511, 122)

In [4]:
df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(41), str(16)
memory usage: 286.2 MB


In [3]:
missing = df.isna().mean().sort_values(ascending=False)
missing.head(20)

COMMONAREA_MEDI             0.698723
COMMONAREA_AVG              0.698723
COMMONAREA_MODE             0.698723
NONLIVINGAPARTMENTS_MODE    0.694330
NONLIVINGAPARTMENTS_AVG     0.694330
NONLIVINGAPARTMENTS_MEDI    0.694330
FONDKAPREMONT_MODE          0.683862
LIVINGAPARTMENTS_MODE       0.683550
LIVINGAPARTMENTS_AVG        0.683550
LIVINGAPARTMENTS_MEDI       0.683550
FLOORSMIN_AVG               0.678486
FLOORSMIN_MODE              0.678486
FLOORSMIN_MEDI              0.678486
YEARS_BUILD_MEDI            0.664978
YEARS_BUILD_MODE            0.664978
YEARS_BUILD_AVG             0.664978
OWN_CAR_AGE                 0.659908
LANDAREA_MEDI               0.593767
LANDAREA_MODE               0.593767
LANDAREA_AVG                0.593767
dtype: float64

In [5]:
features = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
            "DAYS_BIRTH", "DAYS_EMPLOYED", "EXT_SOURCE_1", "EXT_SOURCE_2",
            "EXT_SOURCE_3", "CODE_GENDER", "NAME_CONTRACT_TYPE", "NAME_EDUCATION_TYPE"]
df[features].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
AMT_INCOME_TOTAL,307511.0,NaN,NaN,NaN,168797.919297,237123.146279,25650.0,112500.0,147150.0,202500.0,117000000.0
AMT_CREDIT,307511.0,NaN,NaN,NaN,599025.999706,402490.776996,45000.0,270000.0,513531.0,808650.0,4050000.0
AMT_ANNUITY,307499.0,NaN,NaN,NaN,27108.573909,14493.737315,1615.5,16524.0,24903.0,34596.0,258025.5
AMT_GOODS_PRICE,307233.0,NaN,NaN,NaN,538396.207429,369446.46054,40500.0,238500.0,450000.0,679500.0,4050000.0
DAYS_BIRTH,307511.0,NaN,NaN,NaN,-16036.995067,4363.988632,-25229.0,-19682.0,-15750.0,-12413.0,-7489.0
DAYS_EMPLOYED,307511.0,NaN,NaN,NaN,63815.045904,141275.766519,-17912.0,-2760.0,-1213.0,-289.0,365243.0
EXT_SOURCE_1,134133.0,NaN,NaN,NaN,0.50213,0.211062,0.014568,0.334007,0.505998,0.675053,0.962693
EXT_SOURCE_2,306851.0,NaN,NaN,NaN,0.514393,0.19106,0.0,0.392457,0.565961,0.663617,0.855
EXT_SOURCE_3,246546.0,NaN,NaN,NaN,0.510853,0.194844,0.000527,0.37065,0.535276,0.669057,0.89601
CODE_GENDER,307511,3,F,202448,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
print(df["DAYS_EMPLOYED"].max())
print((df["DAYS_EMPLOYED"] == 365243).mean())

365243
0.18007160719453938


In [7]:
age = df["DAYS_BIRTH"] / -365
age.describe()

count    307511.000000
mean         43.936973
std          11.956133
min          20.517808
25%          34.008219
50%          43.150685
75%          53.923288
max          69.120548
Name: DAYS_BIRTH, dtype: float64

In [8]:
for col in ["CODE_GENDER", "NAME_CONTRACT_TYPE", "NAME_EDUCATION_TYPE"]:
    print(df.groupby(col)["TARGET"].agg(["mean", "count"]), "\n")

                 mean   count
CODE_GENDER                  
F            0.069993  202448
M            0.101419  105059
XNA          0.000000       4 

                        mean   count
NAME_CONTRACT_TYPE                  
Cash loans          0.083459  278232
Revolving loans     0.054783   29279 

                                   mean   count
NAME_EDUCATION_TYPE                            
Academic degree                0.018293     164
Higher education               0.053551   74863
Incomplete higher              0.084850   10277
Lower secondary                0.109277    3816
Secondary / secondary special  0.089399  218391 



In [9]:
df[features + ["TARGET"]].select_dtypes("number").corr()["TARGET"].sort_values()

EXT_SOURCE_3       -0.178919
EXT_SOURCE_2       -0.160472
EXT_SOURCE_1       -0.155317
DAYS_EMPLOYED      -0.044932
AMT_GOODS_PRICE    -0.039645
AMT_CREDIT         -0.030369
AMT_ANNUITY        -0.012817
AMT_INCOME_TOTAL   -0.003982
DAYS_BIRTH          0.078239
TARGET              1.000000
Name: TARGET, dtype: float64

In [10]:
print(df["CODE_GENDER"].value_counts())
print(df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].isna().mean())
print(df.loc[df["DAYS_EMPLOYED"] == 365243, "NAME_INCOME_TYPE"].value_counts())

CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64
EXT_SOURCE_1    0.563811
EXT_SOURCE_2    0.002146
EXT_SOURCE_3    0.198253
dtype: float64
NAME_INCOME_TYPE
Pensioner     55352
Unemployed       22
Name: count, dtype: int64


## Key findings

**Class imbalance.** Only ~8% of clients default, so accuracy is misleading
(always predicting "no default" would score 92%). The model is evaluated
with ROC AUC.

**Missing values.** The external scores have very different coverage:
`EXT_SOURCE_1` is missing for 56% of clients, `EXT_SOURCE_3` for 20% and
`EXT_SOURCE_2` for 0.2%. LightGBM handles missing values natively, so the
API must accept these fields as optional.

**Anomaly in `DAYS_EMPLOYED`.** The value 365243 appears in ~18% of rows.
Almost all of these clients are pensioners (55,352 of 55,374), so the value
means "not employed because retired". It is treated as missing in the
baseline model; a dedicated `IS_PENSIONER` feature could be added later.

**Rare category in `CODE_GENDER`.** Only 4 clients have the value `XNA`,
too few for the model to learn from. The API will accept only `F` and `M`.

**Strongest predictors.** The three `EXT_SOURCE` scores show the strongest
(negative) correlation with default: higher external scores mean lower risk.